
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>

# Lab - Backfilling and Querying Multi-Turn Conversations
#### Duration: 35 minutes

## Overview

This lab focuses on production monitoring, backfill, and querying traces stored in Unity Catalog.

In this lab, you manage a production travel agent. The setup script creates an MLflow experiment that stores its traces in Unity Catalog, logs multi-turn conversation traces to it, and deploys a `ConversationCompleteness` judge. Your job is to inspect the existing setup, run the monitoring job, add a new scorer via backfill, query the traces and their assessments with SQL, and clean up when done.

## Learning Objectives

1. Inspect an MLflow experiment with deployed judges and run the trace metrics computation job
2. Backfill a new scorer on historical traces using `BackfillScorerConfig`
3. Query traces and assessments stored in Unity Catalog with SQL
4. Stop and delete scorers using management APIs

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div>
<strong style="color: #0d47a1; font-size: 1.1em;">Jobs Created in This Lab</strong>
<p style="margin: 8px 0 0 0; color: #333;">As you work through this lab, two types of Lakeflow jobs will be created. You can find them in <strong>Jobs & Pipelines</strong>:</p>
<ul style="margin: 8px 0 0 16px; color: #333;">
<li><strong>[experiment_id] Trace Metrics Computation Job</strong>: created by the setup when it deploys the <code>ConversationCompleteness</code> judge via <code>.start()</code>. Runs scorers on incoming traces.</li>
<li><strong>[experiment_id] Metric Backfill Computation Job</strong>: created when you call <code>backfill_scorers()</code>. Retroactively evaluates historical traces.</li>
</ul>
</div>
</div>

## REQUIRED - SELECT A COMPUTE ENVIRONMENT

<div style="border-left: 4px solid #f44336; background: #ffebee; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #c62828; font-size: 1.1em;">Select Compute</strong>
<p style="margin: 8px 0 0 0; color: #333;">Before starting this notebook, select the required compute environment listed below.</p>
<ul style="margin: 12px 0 0 16px; color: #333;">
<li><strong>Serverless Compute, Version 6</strong>: How to select an environment version (<a href="https://docs.databricks.com/aws/en/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976d2; text-decoration: underline;">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976d2; text-decoration: underline;">Azure</a> | <a href="https://docs.databricks.com/gcp/en/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976d2; text-decoration: underline;">GCP</a>)</li>
<li><strong>SQL Warehouse</strong>: The setup uses <strong>shared_warehouse</strong> to write and read the traces stored in Unity Catalog. You need <strong>CAN USE</strong> on it.</li>
</ul>
<p style="margin: 8px 0 0 0; color: #333;"><strong>NOTE:</strong> This notebook was <strong>developed and tested using Serverless V6</strong>. Other compute options may work but are not guaranteed to behave the same or support all features demonstrated.</p>
</div>
</div>
</div>

## A. Setup

In [0]:
%run ./Includes/Classroom-Setup-07-Lab

## B. Metric Backfill

<div style="background: #1B5162; color: white; padding: 20px 24px; border-radius: 10px; margin: 12px 0;">
<div style="font-size: 11pt; text-transform: uppercase; letter-spacing: 1px; opacity: 0.7; margin-bottom: 6px;">Scenario</div>
<div style="font-size: 14pt; line-height: 1.5;">The setup script created an MLflow experiment with multi-turn conversation traces and deployed a <strong>ConversationCompleteness</strong> judge. Your first task is to verify the setup, run the monitoring job, and inspect the results. Then you will add a new <strong>Safety</strong> scorer and backfill it on the historical traces.</div>
</div>

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #0d47a1; font-size: 1.1em;">Experimental</strong>
<p style="margin: 8px 0 0 0; color: #333;">The <strong>ConversationCompleteness</strong> judge in this lab uses multi-turn evaluation, which is Experimental. Its API and behavior might change in future releases.</p>
</div>
</div>
</div>

### B1. Inspect the Experiment and Run the Deployed Job

The setup script has already:
- Created an MLflow experiment with 3 multi-turn conversation sessions (9 traces total)
- Registered and started a `ConversationCompleteness` judge
- Created the background trace metrics computation job

Your first step is to verify this in the UI, run the job, and see the initial assessments. Click each step below to expand.

<div style="border-left: 4px solid #f44336; background: #ffebee; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div>
<strong style="color: #c62828; font-size: 1.1em;">Default Conversation Timing</strong>
<p style="margin: 8px 0 0 0; color: #333;">Multi-turn judges run after a conversation is considered complete. By default, no new traces with that session ID for <strong>5 minutes</strong>. Make sure the setup script was run at least 5 minutes ago.</p>
</div>
</div>

<br></br>

<div style="max-width: 900px; margin: 0 auto; padding: 4px 6px; box-sizing: border-box; font-family: sans-serif;">

<style>

.lab5-acc-item { background:#fff; border:2px solid #C6DCEA; box-sizing:border-box; border-radius:12px; box-shadow:0 2px 8px rgba(27,49,57,.07); margin-bottom:10px; overflow:hidden; transition:border-color 0.2s, box-shadow 0.2s; }
.lab5-acc-item.open { border-color:#1B5162; box-shadow:0 2px 10px rgba(27,49,57,.10); }
.lab5-acc-header { display:flex; align-items:center; gap:14px; padding:14px 18px; cursor:pointer; user-select:none; background:#fff; transition:background 0.15s; }
.lab5-acc-header:hover { background:#F9F7F4; }
.lab5-acc-num { width:30px; height:30px; border-radius:50%; background:#EEEDE9; color:#5A6F77; font-size:12.5pt; font-weight:700; display:flex; align-items:center; justify-content:center; flex-shrink:0; transition:background 0.2s, color 0.2s; }
.lab5-acc-item.open .lab5-acc-num { background:#1B5162; color:#fff; }
.lab5-acc-summary { flex:1; font-size:13.5pt; font-weight:700; color:#0b2026; }
.lab5-acc-chevron { font-size:12pt; color:#9AABB1; transition:transform 0.25s, color 0.2s; }
.lab5-acc-item.open .lab5-acc-chevron { transform:rotate(180deg); color:#1B5162; }
.lab5-acc-body { max-height:0; opacity:0; overflow:hidden; transition:max-height 0.3s ease, opacity 0.25s ease, padding 0.25s ease; padding:0 18px 0 62px; font-size:14pt; line-height:1.7; color:#333; overflow-wrap:anywhere; }
.lab5-acc-item.open .lab5-acc-body { max-height:2000px; opacity:1; padding:2px 18px 18px 62px; }

</style>

<div id="lab5-acc-b1"></div>

</div>

<script>
var LAB5_B1 = [
  {
    summary: 'Open the MLflow Experiment and review the sessions',
    detail: 'Navigate to <strong>Experiments</strong> in the left menu. Search for <strong>backfill-training</strong> and click on it. On the <strong>Traces</strong> tab, select <strong>Sessions</strong> in the Traces/Sessions switcher to see the 3 conversation sessions. Expand each session to review the turns:<br><br><strong>Session 1 (user_101)</strong>: Japan trip planning. The agent gives thorough, complete answers. This is the baseline.<br><strong>Session 2 (user_202)</strong>: Flight booking. The agent gives vague, incomplete answers ("check online travel sites"). The user has to keep asking for specifics.<br><strong>Session 3 (user_303)</strong>: Hotel cancellation. The agent ignores the request and suggests tourism tips. The user repeats the request with increasing frustration.<br><br>These are the quality issues that multi-turn judges are designed to catch.'
  },
  {
    summary: 'Verify the deployed scorer and its sample rate',
    detail: 'Click <strong>Scorers</strong> under <strong>Evaluation</strong>. You should see <strong>conversation_completeness</strong> listed as an active scorer. This was deployed by the setup script. Note the <strong>sample rate is set to 70%</strong>, meaning about 70% of the sessions will be evaluated on each job run. This is a cost-saving measure for expensive LLM judges.'
  },
  {
    summary: 'Find and run the Trace Metrics Computation Job',
    detail: 'Navigate to <strong>Jobs &amp; Pipelines</strong> and search for <em>Trace Metrics Computation Job</em>. Click <strong>Pause</strong> to stop the automatic schedule, then click <strong>Run now</strong>.'
  },
  {
    summary: 'Inspect the job output',
    detail: 'After the job completes, click the last successful run (green bar). Review the log. You should see traces fetched, session-level scoring for <code>conversation_completeness</code>, and successful evaluations. Because the sample rate is 70%, not all sessions may have assessments.'
  },
  {
    summary: 'View the initial assessments',
    detail: 'Navigate back to the experiment. Select <strong>Sessions</strong> in the Traces/Sessions switcher and expand a session that was evaluated. You should see <strong>ConversationCompleteness</strong> feedback on the first trace of the session. Compare the results: Session 1 (Japan trip) should pass, while Session 2 (vague flight answers) should be flagged as incomplete: exactly the quality issue the users reported.'
  }
];

(function() {
  var container = document.getElementById('lab5-acc-b1');
  LAB5_B1.forEach(function(item, i) {
    var div = document.createElement('div');
    div.className = 'lab5-acc-item' + (i === 0 ? ' open' : '');
    div.innerHTML =
      '<div class="lab5-acc-header" onclick="lab5ToggleB1(' + i + ')">'
      + '<div class="lab5-acc-num">' + (i + 1) + '</div>'
      + '<div class="lab5-acc-summary">' + item.summary + '</div>'
      + '<div class="lab5-acc-chevron">&#9660;</div>'
      + '</div>'
      + '<div class="lab5-acc-body">' + item.detail + '</div>';
    container.appendChild(div);
  });
})();

function lab5ToggleB1(idx) {
  var items = document.querySelectorAll('#lab5-acc-b1 .lab5-acc-item');
  items.forEach(function(el, i) {
    el.classList.toggle('open', i === idx && !el.classList.contains('open'));
  });
}
</script>

### B2. Backfill a New Safety Scorer

Now that you have seen the completeness results, you want to add a **Safety** scorer to check all historical traces for harmful content. Since Safety is a single-turn scorer, you can use `backfill_scorers` with a `BackfillScorerConfig` to retroactively evaluate the existing traces at a custom sample rate.

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div>
<strong style="color: #0d47a1; font-size: 1.1em;">Why backfill instead of just .start()?</strong>
<p style="margin: 8px 0 0 0; color: #333;"><code>.start()</code> only evaluates <strong>new incoming traces</strong>. The 9 traces from the setup already exist. To evaluate them, you need <code>backfill_scorers()</code> which retroactively applies the scorer to historical traces.</p>
</div>
</div>

Running `backfill_scorers()` will create a **[experiment_id] Metric Backfill Computation Job** in Jobs & Pipelines. After running the TODO cell below, navigate to the job in the Jobs UI to inspect the output. You should see the traces being fetched and the Safety scorer evaluating them.

**TODO:** Register and start a `Safety` scorer at `sample_rate=1.0` (safety checks run on every trace), then backfill it on the historical traces using `BackfillScorerConfig` with `sample_rate=1.0`.

In [0]:
# TODO ──────────────────────────────────────────────────────────────────
# 1. Import Safety, ScorerSamplingConfig from mlflow.genai.scorers
# 2. Import backfill_scorers, BackfillScorerConfig from databricks.agents.scorers
# 3. Create a Safety scorer with model=judge_model, then register and start it
#    (name="safety_check", sample_rate=1.0)
# 4. Call backfill_scorers with BackfillScorerConfig(scorer=safety, sample_rate=1.0)
# 5. Print the returned job_id
# ──────────────────────────────────────────────────────────────────────


<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">ANSWER</span> B2. Backfill a New Safety Scorer
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;">

<div class="code-card" style="font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;border:1px solid #DCE0E2;border-radius:8px;overflow:hidden;background:#F9F7F4;max-width:100%;margin:12px 0;">
<div style="display:flex;align-items:center;justify-content:space-between;padding:0.45rem 0.75rem;background:#EEEDE9;border-bottom:1px solid #DCE0E2;">
  <div style="display:inline-flex;align-items:center;gap:0.45rem;">
    <svg width="18" height="18" viewBox="0 0 24 24" fill="none"><path d="M12 2C8 2 8 4.5 8 4.5V7h4v1H6.5S3 8 3 12.5 5.5 17 5.5 17H8v-3.5S7.8 11 10 11h4s2 .1 2-2V4.5S16 2 12 2zm-1 1.8a.9.9 0 110 1.8.9.9 0 010-1.8z" fill="#5A6F77"/><path d="M12 22c4 0 4-2.5 4-2.5V17h-4v-1h5.5s3.5 0 3.5-4.5S18.5 7 18.5 7H16v3.5s.2 2.5-2 2.5h-4s-2-.1-2 2v4.5S8 22 12 22zm1-1.8a.9.9 0 110-1.8.9.9 0 010 1.8z" fill="none" stroke="#5A6F77" stroke-width="1.5"/></svg>
    <span style="font-size:0.72rem;font-weight:600;text-transform:uppercase;letter-spacing:0.1em;color:#5A6F77;">Python</span>
  </div>
  <button style="display:inline-flex;align-items:center;gap:0.3rem;background:none;border:none;color:#5A6F77;cursor:pointer;padding:0.2rem 0.4rem;border-radius:0.25rem;font-size:0.7rem;font-weight:500;" onclick="(function(b){var c=b.closest('.code-card').querySelector('.code-card-body').textContent,t=document.createElement('textarea');t.value=c;t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();var ok=false;try{ok=document.execCommand('copy')}catch(e){}document.body.removeChild(t);if(ok){var l=b.querySelector('.copy-label'),ci=b.querySelector('.copy-icon'),chi=b.querySelector('.check-icon');ci.style.display='none';chi.style.display='';l.textContent='Copied!';b.style.color='#2272B4';setTimeout(function(){ci.style.display='';chi.style.display='none';l.textContent='Copy';b.style.color='#5A6F77'},2000)}})(this)">
    <svg class="copy-icon" fill="none" stroke="currentColor" viewBox="0 0 24 24" style="width:14px;height:14px" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"><rect x="9" y="9" width="13" height="13" rx="2"/><path d="M5 15H4a2 2 0 01-2-2V4a2 2 0 012-2h9a2 2 0 012 2v1"/></svg>
    <svg class="check-icon" style="display:none;width:14px;height:14px" fill="none" stroke="currentColor" viewBox="0 0 24 24" stroke-width="2.5" stroke-linecap="round" stroke-linejoin="round"><polyline points="20 6 9 17 4 12"/></svg>
    <span class="copy-label">Copy</span>
  </button>
</div>
<div class="code-card-body" style="margin:0;padding:1rem;overflow-x:auto;font-family:'SF Mono','Cascadia Code','Fira Code',Consolas,monospace;font-size:0.82rem;line-height:1.55;color:#1B3139;white-space:pre-wrap;word-break:break-word;">from mlflow.genai.scorers import Safety, ScorerSamplingConfig
from databricks.agents.scorers import backfill_scorers, BackfillScorerConfig
<span></span>
# Register and start the Safety scorer
safety = Safety(model=judge_model).register(name="safety_check")
safety = safety.start(sampling_config=ScorerSamplingConfig(sample_rate=1.0))
<span></span>
# Backfill every historical trace
job_id = backfill_scorers(
    scorers=[BackfillScorerConfig(scorer=safety, sample_rate=1.0)]
)
print(f"Backfill job ID: {job_id}")</div>
</div>

</div>
</details>

<details style="margin: 14px 4px; background: #fff; border: 1.5px solid #C6DCEA; border-radius: 14px; padding: 0 18px; box-shadow: 0 2px 8px rgba(27,49,57,.07); font-family: sans-serif;">
<summary style="cursor: pointer; padding: 16px 0; font-weight: 700; font-size: 18px; color: #0b2026; user-select: none;">
<span style="display: inline-block; background: #5A6F77; color: #fff; border-radius: 999px; padding: 3px 12px; font-size: 12px; font-weight: 700; letter-spacing: 0.06em; margin-right: 8px; vertical-align: 2px;">DEEP DIVE</span> What does backfill_scorers() actually do?
</summary>
<div style="padding:0 0 18px;font-size:14pt;line-height:1.7;color:#333">

<p><code>backfill_scorers</code> retroactively applies scorers to historical traces that already exist in the experiment. Per the docs: <em>"You can retroactively apply new or updated scorers to historical traces."</em></p>

<p style="font-weight:500;color:#0b2026;margin:14px 0 6px">Step by step:</p>
<ol style="padding-left:18px">
<li><strong>Resolves the active experiment</strong>. Uses the currently active MLflow experiment, or an explicit <code>experiment_id=</code> parameter</li>
<li><strong>Looks up scorers</strong>. Accepts name strings (uses registered sample rates) or <code>BackfillScorerConfig</code> objects (custom rates per scorer)</li>
<li><strong>Creates a backfill job</strong>. Returns a <code>job_id</code> (string) for status tracking</li>
</ol>

<table style="width:100%;border-collapse:collapse;margin:14px 0;font-size:17px">
<thead><tr><th style="background:#f0f2f7;color:#555;padding:8px 12px;text-align:left;font-weight:500;border-bottom:0.5px solid #dde0e8">Scheduled job (.start())</th><th style="background:#f0f2f7;color:#555;padding:8px 12px;text-align:left;font-weight:500;border-bottom:0.5px solid #dde0e8">Backfill (backfill_scorers())</th></tr></thead>
<tbody>
<tr><td style="padding:8px 12px;border-bottom:0.5px solid #dde0e8">Evaluates new incoming traces on a recurring schedule</td><td style="padding:8px 12px;border-bottom:0.5px solid #dde0e8">Evaluates historical traces retroactively, once</td></tr>
<tr><td style="padding:8px 12px;border-bottom:0.5px solid #dde0e8;background:#f8f8f8">Ongoing</td><td style="padding:8px 12px;border-bottom:0.5px solid #dde0e8;background:#f8f8f8">One-shot</td></tr>
</tbody></table>

<p style="font-weight:500;color:#0b2026;margin:14px 0 6px">Important constraints:</p>
<ul style="padding-left:18px">
<li>Scorers must already be registered and started in the experiment</li>
<li>Without <code>start_time</code> or <code>end_time</code>, the backfill covers only the last seven days. Pass an explicit <code>start_time</code> to evaluate older traces</li>
</ul>

<p style="font-size:14px;color:#999;margin-top:12px"><em>Source: Backfill historical traces with scorers (<a href="https://docs.databricks.com/aws/en/mlflow3/genai/eval-monitor/backfill-scorers" style="color:#378ADD">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/mlflow3/genai/eval-monitor/backfill-scorers" style="color:#378ADD">Azure</a> | <a href="https://docs.databricks.com/gcp/en/mlflow3/genai/eval-monitor/backfill-scorers" style="color:#378ADD">GCP</a>)</em></p>

</div>
</details>

### B3. Verify Assessments on Traces

Run the cell below to search for recent traces and confirm that the Safety backfill assessments have been attached. The backfill runs as a separate job and takes a few minutes. If the `assessments` column is still empty, wait for the **Metric Backfill Computation Job** run to finish in Jobs & Pipelines, then rerun the cell.

In [0]:
# Verify: Search traces and check for assessments
import mlflow

experiment_id = mlflow.get_experiment_by_name(mlflow_experiment_path).experiment_id

traces_df = mlflow.search_traces(
    locations=[experiment_id],
    max_results=10
)

# Show trace IDs and assessments
display(traces_df[["trace_id", "assessments"]])

## C. Traces in Unity Catalog

<div style="background: #1B5162; color: white; padding: 20px 24px; border-radius: 10px; margin: 12px 0;">
<div style="font-size: 11pt; text-transform: uppercase; letter-spacing: 1px; opacity: 0.7; margin-bottom: 6px;">Scenario</div>
<div style="font-size: 14pt; line-height: 1.5;">The data engineering team wants to build AI/BI dashboards over agent traces and their assessments. This experiment already stores its traces in Unity Catalog, so the data is in governed Delta tables in your schema. There is nothing to copy or sync.</div>
</div>

### C1. Explore the Trace Tables

The setup created this experiment with `trace_location=UnityCatalog(...)`, so MLflow writes every trace and assessment straight into Delta tables named `backfill_traces_*` in your schema. The older pattern copied traces out of the experiment's own store with a trace archive job (`enable_databricks_trace_archival`). Storing traces in Unity Catalog replaces it:

<table style="width:100%;border-collapse:collapse;margin:14px 0;font-size:15px">
<thead><tr><th style="background:#f0f2f7;color:#555;padding:8px 12px;text-align:left;font-weight:500;border-bottom:0.5px solid #dde0e8"></th><th style="background:#f0f2f7;color:#555;padding:8px 12px;text-align:left;font-weight:500;border-bottom:0.5px solid #dde0e8">Trace archive job (older)</th><th style="background:#f0f2f7;color:#555;padding:8px 12px;text-align:left;font-weight:500;border-bottom:0.5px solid #dde0e8">Traces in Unity Catalog</th></tr></thead>
<tbody>
<tr><td style="padding:8px 12px;border-bottom:0.5px solid #dde0e8;font-weight:600">Storage</td><td style="padding:8px 12px;border-bottom:0.5px solid #dde0e8">A copy in a Delta table, next to the experiment's own store</td><td style="padding:8px 12px;border-bottom:0.5px solid #dde0e8">One copy: the Delta tables are the trace store</td></tr>
<tr><td style="padding:8px 12px;border-bottom:0.5px solid #dde0e8;font-weight:600">Freshness</td><td style="padding:8px 12px;border-bottom:0.5px solid #dde0e8">Lags until the archive job runs</td><td style="padding:8px 12px;border-bottom:0.5px solid #dde0e8">Queryable as traces are written</td></tr>
<tr><td style="padding:8px 12px;border-bottom:0.5px solid #dde0e8;font-weight:600">Access</td><td style="padding:8px 12px;border-bottom:0.5px solid #dde0e8">Experiment permissions, plus separate grants on the copy</td><td style="padding:8px 12px;border-bottom:0.5px solid #dde0e8">Unity Catalog grants, lineage, and auditing, like any other table</td></tr>
<tr><td style="padding:8px 12px;border-bottom:0.5px solid #dde0e8;font-weight:600">Operations</td><td style="padding:8px 12px;border-bottom:0.5px solid #dde0e8">An extra job to enable, run, and disable</td><td style="padding:8px 12px;border-bottom:0.5px solid #dde0e8">No extra job</td></tr>
</tbody></table>

Click each step below to expand.

<br></br>

<div style="max-width: 900px; margin: 0 auto; padding: 4px 6px; box-sizing: border-box; font-family: sans-serif;">

<style>

.lab5-acc2-item { background:#fff; border:2px solid #C6DCEA; box-sizing:border-box; border-radius:12px; box-shadow:0 2px 8px rgba(27,49,57,.07); margin-bottom:10px; overflow:hidden; transition:border-color 0.2s, box-shadow 0.2s; }
.lab5-acc2-item.open { border-color:#1B5162; box-shadow:0 2px 10px rgba(27,49,57,.10); }
.lab5-acc2-header { display:flex; align-items:center; gap:14px; padding:14px 18px; cursor:pointer; user-select:none; background:#fff; transition:background 0.15s; }
.lab5-acc2-header:hover { background:#F9F7F4; }
.lab5-acc2-num { width:30px; height:30px; border-radius:50%; background:#EEEDE9; color:#5A6F77; font-size:12.5pt; font-weight:700; display:flex; align-items:center; justify-content:center; flex-shrink:0; transition:background 0.2s, color 0.2s; }
.lab5-acc2-item.open .lab5-acc2-num { background:#1B5162; color:#fff; }
.lab5-acc2-summary { flex:1; font-size:13.5pt; font-weight:700; color:#0b2026; }
.lab5-acc2-chevron { font-size:12pt; color:#9AABB1; transition:transform 0.25s, color 0.2s; }
.lab5-acc2-item.open .lab5-acc2-chevron { transform:rotate(180deg); color:#1B5162; }
.lab5-acc2-body { max-height:0; opacity:0; overflow:hidden; transition:max-height 0.3s ease, opacity 0.25s ease, padding 0.25s ease; padding:0 18px 0 62px; font-size:14pt; line-height:1.7; color:#333; overflow-wrap:anywhere; }
.lab5-acc2-item.open .lab5-acc2-body { max-height:2000px; opacity:1; padding:2px 18px 18px 62px; }

</style>

<div id="lab5-acc-c1b"></div>

</div>

<script>
var LAB5_C1B = [
  {
    summary: 'Open your schema in Unity Catalog',
    detail: 'Open <strong>Catalog</strong> in the left menu and navigate to your catalog and the <strong>agent_monitoring</strong> schema. You should see four tables, <strong>backfill_traces_otel_spans</strong>, <strong>backfill_traces_otel_logs</strong>, <strong>backfill_traces_otel_annotations</strong>, and <strong>backfill_traces_otel_metrics</strong>, and two views, <strong>backfill_traces_trace_unified</strong> and <strong>backfill_traces_trace_metadata</strong>.'
  },
  {
    summary: 'Preview the trace data',
    detail: 'Click <strong>backfill_traces_trace_unified</strong>, select a SQL warehouse at the top of the page, and click <strong>Sample Data</strong>. Each row is one trace, with its request, response, duration, metadata, spans, and assessments. The <strong>backfill_traces_trace_metadata</strong> view has the same traces without span data, so it is faster to query for assessments.'
  },
  {
    summary: 'Check who can access the traces',
    detail: 'Click the <strong>Permissions</strong> tab. Access to traces is managed with Unity Catalog grants, the same way as any other table. To read and write traces, users need <strong>SELECT</strong> and <strong>MODIFY</strong> on the trace tables.'
  }
];

(function() {
  var container = document.getElementById('lab5-acc-c1b');
  LAB5_C1B.forEach(function(item, i) {
    var div = document.createElement('div');
    div.className = 'lab5-acc2-item' + (i === 0 ? ' open' : '');
    div.innerHTML =
      '<div class="lab5-acc2-header" onclick="lab5ToggleC1(' + i + ')">'
      + '<div class="lab5-acc2-num">' + (i + 1) + '</div>'
      + '<div class="lab5-acc2-summary">' + item.summary + '</div>'
      + '<div class="lab5-acc2-chevron">&#9660;</div>'
      + '</div>'
      + '<div class="lab5-acc2-body">' + item.detail + '</div>';
    container.appendChild(div);
  });
})();

function lab5ToggleC1(idx) {
  var items = document.querySelectorAll('#lab5-acc-c1b .lab5-acc2-item');
  items.forEach(function(el, i) {
    el.classList.toggle('open', i === idx && !el.classList.contains('open'));
  });
}
</script>

### C2. Query Assessments with SQL

Because the assessments are stored with the traces, one SQL query shows every judge's result for every session. Run it after the jobs from section B finish, so the results are there to query. This is the kind of query an AI/BI dashboard would run.

**TODO:** Query `{catalog_name}.{schema_name}.backfill_traces_trace_metadata` and list each session's user, assessment name, and value. Expand the `assessments` array with `LATERAL VIEW explode(...)` and skip assessments whose scorer returned an error.

In [0]:
# TODO ──────────────────────────────────────────────────────────────────
# Use spark.sql() on {catalog_name}.{schema_name}.backfill_traces_trace_metadata
#   - Session ID: trace_metadata:['mlflow.trace.session']::STRING
#   - User ID:    trace_metadata:['mlflow.trace.user']::STRING
#   - Expand assessments with LATERAL VIEW explode(assessments) t AS a
#   - Select a.assessment_name and a.feedback.value
#   - Keep rows where a.feedback.error IS NULL
# Display the result with display()
# ──────────────────────────────────────────────────────────────────────


<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">ANSWER</span> C2. Query Assessments with SQL
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;">

<div class="code-card" style="font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;border:1px solid #DCE0E2;border-radius:8px;overflow:hidden;background:#F9F7F4;max-width:100%;margin:12px 0;">
<div style="display:flex;align-items:center;justify-content:space-between;padding:0.45rem 0.75rem;background:#EEEDE9;border-bottom:1px solid #DCE0E2;">
  <div style="display:inline-flex;align-items:center;gap:0.45rem;">
    <svg width="18" height="18" viewBox="0 0 24 24" fill="none"><path d="M12 2C8 2 8 4.5 8 4.5V7h4v1H6.5S3 8 3 12.5 5.5 17 5.5 17H8v-3.5S7.8 11 10 11h4s2 .1 2-2V4.5S16 2 12 2zm-1 1.8a.9.9 0 110 1.8.9.9 0 010-1.8z" fill="#5A6F77"/><path d="M12 22c4 0 4-2.5 4-2.5V17h-4v-1h5.5s3.5 0 3.5-4.5S18.5 7 18.5 7H16v3.5s.2 2.5-2 2.5h-4s-2-.1-2 2v4.5S8 22 12 22zm1-1.8a.9.9 0 110-1.8.9.9 0 010 1.8z" fill="none" stroke="#5A6F77" stroke-width="1.5"/></svg>
    <span style="font-size:0.72rem;font-weight:600;text-transform:uppercase;letter-spacing:0.1em;color:#5A6F77;">Python</span>
  </div>
  <button style="display:inline-flex;align-items:center;gap:0.3rem;background:none;border:none;color:#5A6F77;cursor:pointer;padding:0.2rem 0.4rem;border-radius:0.25rem;font-size:0.7rem;font-weight:500;" onclick="(function(b){var c=b.closest('.code-card').querySelector('.code-card-body').textContent,t=document.createElement('textarea');t.value=c;t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();var ok=false;try{ok=document.execCommand('copy')}catch(e){}document.body.removeChild(t);if(ok){var l=b.querySelector('.copy-label'),ci=b.querySelector('.copy-icon'),chi=b.querySelector('.check-icon');ci.style.display='none';chi.style.display='';l.textContent='Copied!';b.style.color='#2272B4';setTimeout(function(){ci.style.display='';chi.style.display='none';l.textContent='Copy';b.style.color='#5A6F77'},2000)}})(this)">
    <svg class="copy-icon" fill="none" stroke="currentColor" viewBox="0 0 24 24" style="width:14px;height:14px" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"><rect x="9" y="9" width="13" height="13" rx="2"/><path d="M5 15H4a2 2 0 01-2-2V4a2 2 0 012-2h9a2 2 0 012 2v1"/></svg>
    <svg class="check-icon" style="display:none;width:14px;height:14px" fill="none" stroke="currentColor" viewBox="0 0 24 24" stroke-width="2.5" stroke-linecap="round" stroke-linejoin="round"><polyline points="20 6 9 17 4 12"/></svg>
    <span class="copy-label">Copy</span>
  </button>
</div>
<div class="code-card-body" style="margin:0;padding:1rem;overflow-x:auto;font-family:'SF Mono','Cascadia Code','Fira Code',Consolas,monospace;font-size:0.82rem;line-height:1.55;color:#1B3139;white-space:pre-wrap;word-break:break-word;">display(spark.sql(f"""
    SELECT trace_metadata:['mlflow.trace.session']::STRING AS session_id,
           trace_metadata:['mlflow.trace.user']::STRING AS user_id,
           a.assessment_name,
           a.feedback.value AS value
    FROM {catalog_name}.{schema_name}.backfill_traces_trace_metadata
    LATERAL VIEW explode(assessments) t AS a
    WHERE a.feedback.error IS NULL
    ORDER BY user_id, a.assessment_name
"""))</div>
</div>

</div>
</details>

## D. Cleanup: Stop and Delete All Scorers
Stop all scorers, then delete them. Use `list_scorers()` to enumerate, `.stop()` to pause, and `delete_scorer()` to remove permanently. Stopping first is optional, since `delete_scorer()` also removes a running scorer, but it shows both lifecycle steps. Historical assessments remain on their traces even after deletion.

In [0]:
# TODO ──────────────────────────────────────────────────────────────────
# 1. List all scorers
# 2. Stop each one (reassign: .stop() returns a new scorer)
# 3. Delete each one using delete_scorer(name=...)
# 4. Verify the list is empty
#
# Hint: from mlflow.genai.scorers import list_scorers, delete_scorer
# ──────────────────────────────────────────────────────────────────────


<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">ANSWER</span> D. Cleanup
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;">

<div class="code-card" style="font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;border:1px solid #DCE0E2;border-radius:8px;overflow:hidden;background:#F9F7F4;max-width:100%;margin:12px 0;">
<div style="display:flex;align-items:center;justify-content:space-between;padding:0.45rem 0.75rem;background:#EEEDE9;border-bottom:1px solid #DCE0E2;">
  <div style="display:inline-flex;align-items:center;gap:0.45rem;">
    <svg width="18" height="18" viewBox="0 0 24 24" fill="none"><path d="M12 2C8 2 8 4.5 8 4.5V7h4v1H6.5S3 8 3 12.5 5.5 17 5.5 17H8v-3.5S7.8 11 10 11h4s2 .1 2-2V4.5S16 2 12 2zm-1 1.8a.9.9 0 110 1.8.9.9 0 010-1.8z" fill="#5A6F77"/><path d="M12 22c4 0 4-2.5 4-2.5V17h-4v-1h5.5s3.5 0 3.5-4.5S18.5 7 18.5 7H16v3.5s.2 2.5-2 2.5h-4s-2-.1-2 2v4.5S8 22 12 22zm1-1.8a.9.9 0 110-1.8.9.9 0 010 1.8z" fill="none" stroke="#5A6F77" stroke-width="1.5"/></svg>
    <span style="font-size:0.72rem;font-weight:600;text-transform:uppercase;letter-spacing:0.1em;color:#5A6F77;">Python</span>
  </div>
  <button style="display:inline-flex;align-items:center;gap:0.3rem;background:none;border:none;color:#5A6F77;cursor:pointer;padding:0.2rem 0.4rem;border-radius:0.25rem;font-size:0.7rem;font-weight:500;" onclick="(function(b){var c=b.closest('.code-card').querySelector('.code-card-body').textContent,t=document.createElement('textarea');t.value=c;t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();var ok=false;try{ok=document.execCommand('copy')}catch(e){}document.body.removeChild(t);if(ok){var l=b.querySelector('.copy-label'),ci=b.querySelector('.copy-icon'),chi=b.querySelector('.check-icon');ci.style.display='none';chi.style.display='';l.textContent='Copied!';b.style.color='#2272B4';setTimeout(function(){ci.style.display='';chi.style.display='none';l.textContent='Copy';b.style.color='#5A6F77'},2000)}})(this)">
    <svg class="copy-icon" fill="none" stroke="currentColor" viewBox="0 0 24 24" style="width:14px;height:14px" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"><rect x="9" y="9" width="13" height="13" rx="2"/><path d="M5 15H4a2 2 0 01-2-2V4a2 2 0 012-2h9a2 2 0 012 2v1"/></svg>
    <svg class="check-icon" style="display:none;width:14px;height:14px" fill="none" stroke="currentColor" viewBox="0 0 24 24" stroke-width="2.5" stroke-linecap="round" stroke-linejoin="round"><polyline points="20 6 9 17 4 12"/></svg>
    <span class="copy-label">Copy</span>
  </button>
</div>
<div class="code-card-body" style="margin:0;padding:1rem;overflow-x:auto;font-family:'SF Mono','Cascadia Code','Fira Code',Consolas,monospace;font-size:0.82rem;line-height:1.55;color:#1B3139;white-space:pre-wrap;word-break:break-word;">from mlflow.genai.scorers import list_scorers, delete_scorer
<span></span>
scorers = list_scorers()
print(f"Stopping and deleting {len(scorers)} scorers:\n")
<span></span>
for s in scorers:
    print(f"  {s.name}...", end=" ")
    stopped = s.stop()
    delete_scorer(name=stopped.name)
    print(f"stopped (rate={stopped.sample_rate}) and deleted")
<span></span>
# Verify
remaining = list_scorers()
print(f"\nRemaining scorers: {len(remaining)}")</div>
</div>

</div>
</details>

## Conclusion

1. **Metric backfill** (`databricks.agents.scorers.backfill_scorers`) retroactively evaluates historical traces with per-scorer sample rate and time range controls
2. **Two backfill modes**: pass scorer name strings (uses registered rates) or `BackfillScorerConfig` objects (custom rates)
3. **Traces in Unity Catalog** (`trace_location=UnityCatalog(...)`) are governed Delta tables from the start: query traces and assessments with SQL, manage access with Unity Catalog grants, and skip the older trace archive job
4. **Management APIs** (`list_scorers`, `get_scorer`, `delete_scorer`) let you audit, update, and clean up scorers in production
5. **Cleanup matters**: always stop and delete scorers when done to avoid unnecessary evaluation costs

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>
